# A one-dimensional Kalman filter: a slow baseline and local variation

This notebook explores the identity-measurement random-walk model discussed for AEther. The hidden state and measurement use the **same units and coordinate**.

```text
x[k] = x[k-1] + w[k]     w[k] ~ Normal(0, q)
y[k] = x[k]   + v[k]     v[k] ~ Normal(0, r)
```

- **q** is the variance of the hidden baseline's change **per frame**. Its typical step size is sqrt(q).
- **r** is the variance of local measurement deviations. Their typical size is sqrt(r).
- **p** is the filter's uncertainty about the current state. It is a third, different variance.
- The Gaussian noises are independent across frames and of each other. The initial state is Gaussian in the filter model.

The default experiment uses the same q and r for the simulator and filter. Uncheck **Match simulator** to distinguish what actually happens from what the filter assumes.

**Run all cells.** Dependencies: numpy, matplotlib, ipywidgets, and IPython. Use JupyterLab, Jupyter Notebook, or a notebook editor with ipywidgets support. If needed, uncomment and run the installation cell, then restart the kernel.


In [1]:
# Uncomment if these packages are missing in your notebook kernel:
# %pip install numpy matplotlib ipywidgets


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


## The update, one frame at a time

```text
m_pred = m_previous
p_pred = p_previous + q

gain  = p_pred / (p_pred + r)
m_new = m_pred + gain * (y - m_pred)
p_new = (1 - gain) * p_pred
```

The **innovation**, y - m_pred, is the discrepancy from the predicted measurement. The gain is the fraction used to move the estimate.

At frame 0 we assimilate the first measurement into the initial prior, without adding q: no transition has happened yet. The implementation uses the equivalent scalar Joseph formula for the covariance update to improve numerical stability.

With fixed positive q and r, the gain approaches a constant a:

```text
q/r = a*a / (1-a)
a = 2*q / (q + sqrt(q*q + 4*q*r))
```

At steady state, the mean update is an exponential smoother. For small q/r, a is approximately sqrt(q/r). q must be rescaled if the duration of a frame changes; it is not a continuous-time rate here.


In [3]:
def simulate(n, q, r, seed, scenario):
    """Use fixed standard-normal draws so slider changes remain comparable."""
    rng = np.random.default_rng(seed)
    z_step = rng.standard_normal(n)
    z_meas = rng.standard_normal(n)
    w = np.sqrt(q) * z_step
    w[0] = 0.0
    x = np.cumsum(w)  # The simulated state starts at exactly zero.
    if scenario == "Abrupt change":
        x[n // 2:] += 2.0  # Intentional violation of the small-step model.
    v = np.sqrt(r) * z_meas
    return x, x + v, w, v


def kalman_1d(y, q, r, m0=0.0, p0=1.0):
    """Random-walk Kalman filter, H=1, with fixed q and r > 0."""
    y = np.asarray(y, dtype=float)
    m = np.empty_like(y)
    p = np.empty_like(y)
    pred_m = np.empty_like(y)
    pred_p = np.empty_like(y)
    gain = np.empty_like(y)
    current_m, current_p = float(m0), float(p0)
    for k, measurement in enumerate(y):
        pp = current_p + (q if k > 0 else 0.0)
        a = pp / (pp + r)
        pred_m[k], pred_p[k], gain[k] = current_m, pp, a
        current_m += a * (measurement - current_m)
        current_p = (1.0 - a)**2 * pp + a*a*r
        m[k], p[k] = current_m, current_p
    return dict(m=m, p=p, pred_m=pred_m, pred_p=pred_p, gain=gain)


def steady_gain(q, r):
    return 2.0*q / (q + np.sqrt(q*q + 4.0*q*r))


In [4]:
N = 300
style = {"description_width": "145px"}
wide = widgets.Layout(width="470px")


def log_slider(label, value, low, high):
    return widgets.FloatLogSlider(
        description=label, value=value, base=10, min=low, max=high,
        step=0.1, readout_format=".3g", continuous_update=False,
        style=style, layout=wide,
    )


q_filter = log_slider("q assumed / frame", 0.01, -6, 1)
r_filter = log_slider("r assumed", 0.25, -4, 1)
matched = widgets.Checkbox(value=True, description="Match simulator to assumed q and r",
                           layout=widgets.Layout(width="470px"))
q_actual = log_slider("q actual / frame", 0.01, -6, 1)
r_actual = log_slider("r actual", 0.25, -4, 1)
p_initial = log_slider("Initial variance p0", 1.0, -6, 2)
m_initial = widgets.FloatSlider(
    description="Initial mean m0", value=0.0, min=-5, max=5, step=0.1,
    continuous_update=False, style=style, layout=wide,
)
scenario = widgets.Dropdown(
    description="Signal", options=["Random walk", "Abrupt change"],
    style=style, layout=wide,
)
seed = widgets.IntSlider(
    description="Random seed", value=7, min=0, max=50,
    continuous_update=False, style=style, layout=wide,
)
frame = widgets.IntSlider(
    description="Inspect frame k", value=80, min=0, max=N-1,
    continuous_update=False, style=style, layout=wide,
)
notes = widgets.HTML()
step_detail = widgets.HTML()
plot_output = widgets.Output()


def render(change=None):
    linked = matched.value
    q_actual.disabled = linked
    r_actual.disabled = linked
    q, r = q_filter.value, r_filter.value
    qa = q if linked else q_actual.value
    ra = r if linked else r_actual.value
    x, y, w, v = simulate(N, qa, ra, seed.value, scenario.value)
    result = kalman_1d(y, q, r, m_initial.value, p_initial.value)
    m, p, a = result["m"], result["p"], result["gain"]
    pp, mp = result["pred_p"], result["pred_m"]
    k = frame.value
    a_inf = steady_gain(q, r)
    tau = -1.0 / np.log1p(-a_inf)
    error = m - x
    rmse = np.sqrt(np.mean(error**2))
    coverage = np.mean(np.abs(error) <= 1.96*np.sqrt(p))
    mismatch = ""
    if scenario.value == "Abrupt change":
        mismatch = (" <b>The +2 jump at frame 150 is deliberately outside the "
                    "ordinary small-step experiment.</b>")
    notes.value = (
        f"<p><b>Simulator:</b> sqrt(q) = {np.sqrt(qa):.4g} state units/frame; "
        f"sqrt(r) = {np.sqrt(ra):.4g} state units. "
        f"<b>Filter:</b> q/r = {q/r:.4g}; steady gain = {a_inf:.4g}; "
        f"steady smoothing time = {tau:.2f} frames (one e-fold).{mismatch}</p>"
        f"<p>Realized state RMSE = {rmse:.4g}; fraction inside the nominal 95% "
        f"state interval = {coverage:.1%}. This is one correlated trajectory, "
        f"so its coverage need not be exactly 95%.</p>"
    )
    transition_q = q if k > 0 else 0.0
    innovation = y[k] - mp[k]
    previous_p = p[k-1] if k > 0 else p_initial.value
    step_detail.value = f"""
    <div style="padding:12px;border:1px solid #aab3c0;border-radius:6px">
    <b>Frame {k}: prediction → measurement → update</b>
    <pre style="white-space:pre-wrap">
    Predicted mean         = {mp[k]: .6f}
    Predicted variance     = {previous_p:.6g} + {transition_q:.6g} = {pp[k]:.6g}
    Measurement            = {y[k]: .6f}
    Innovation             = {y[k]:.6f} - ({mp[k]:.6f}) = {innovation:.6f}
    Gain                   = {pp[k]:.6g} / ({pp[k]:.6g} + {r:.6g}) = {a[k]:.6f}
    Updated mean           = {mp[k]:.6f} + {a[k]:.6f} * ({innovation:.6f}) = {m[k]:.6f}
    Updated variance       = (1 - {a[k]:.6f}) * {pp[k]:.6g} = {p[k]:.6g}
    Hidden state (simulator only) = {x[k]: .6f}
    </pre>
    </div>
    """
    t = np.arange(N)
    with plot_output:
        plot_output.clear_output(wait=True)
        fig = plt.figure(figsize=(12, 10), constrained_layout=True)
        gs = fig.add_gridspec(3, 2, height_ratios=[1.7, 1, 1])
        track = fig.add_subplot(gs[0, :])
        gain_ax = fig.add_subplot(gs[1, 0])
        variance_ax = fig.add_subplot(gs[1, 1])
        step_ax = fig.add_subplot(gs[2, 0])
        noise_ax = fig.add_subplot(gs[2, 1])

        track.scatter(t, y, s=9, color="#929ba8", alpha=0.45,
                      label="measurement y")
        track.plot(t, x, color="#202b38", lw=1.8, label="hidden state x")
        track.plot(t, m, color="#1675c5", lw=2, label="filtered estimate m")
        track.fill_between(t, m-1.96*np.sqrt(p), m+1.96*np.sqrt(p),
                           color="#1675c5", alpha=0.16,
                           label="nominal 95% state interval")
        track.scatter([k], [m[k]], color="#d46714", zorder=5, s=35)
        track.set(title="A slow baseline, noisy local observations, and the inferred state",
                  ylabel="state units")
        track.legend(loc="upper left", ncol=2, fontsize=9)

        gain_ax.plot(t, a, color="#1675c5", label="Kalman gain")
        gain_ax.axhline(a_inf, color="#d46714", ls="--", label="steady gain")
        gain_ax.set(title="How much of each innovation enters the estimate",
                    ylabel="gain (fraction)", ylim=(0, 1.02))
        gain_ax.legend(fontsize=9)

        variance_ax.semilogy(t, pp, color="#8b62aa", label="predicted variance")
        variance_ax.semilogy(t, p, color="#1675c5", label="updated variance")
        variance_ax.axhline(r, color="#858585", ls=":", label="assumed r")
        variance_ax.set(title="Uncertainty increases, then observations reduce it",
                        ylabel="variance (state units squared)")
        variance_ax.legend(fontsize=8)

        for ax, draws, sd, color, title in [
            (step_ax, w, np.sqrt(qa), "#3e946b", "Actual random state increments w"),
            (noise_ax, v, np.sqrt(ra), "#bc6855", "Actual measurement deviations v"),
        ]:
            ax.axhspan(-sd, sd, color=color, alpha=0.12, label="±1 actual standard deviation")
            ax.plot(t, draws, color=color, lw=0.8)
            ax.axhline(0, color="#777777", lw=0.6)
            ax.set(title=title, ylabel="state units")
            ax.legend(fontsize=8)
        if scenario.value == "Abrupt change":
            step_ax.set_title("Random increments w (the extra +2 jump is excluded)")
            track.axvline(N//2, color="#bc6855", ls=":", lw=1)
        for ax in [track, gain_ax, variance_ax, step_ax, noise_ax]:
            ax.axvline(k, color="#d46714", alpha=0.65, lw=1)
            ax.set_xlim(0, N-1)
            ax.set_xlabel("frame k")
            ax.grid(alpha=0.15)
        plt.show()
        plt.close(fig)


for control in [q_filter, r_filter, matched, q_actual, r_actual,
                p_initial, m_initial, scenario, seed, frame]:
    control.observe(render, names="value")

display(widgets.VBox([
    widgets.HBox([
        widgets.VBox([q_filter, r_filter, matched, q_actual, r_actual]),
        widgets.VBox([scenario, p_initial, m_initial, seed, frame]),
    ]),
    notes, step_detail, plot_output,
]))
render()


## Experiments to try

1. **Matched model: change q.** Keep r fixed and increase q. The hidden baseline wanders more rapidly and the filter increases its gain. Decrease q and watch both become steadier.
2. **Matched model: change r.** Keep q fixed and increase r. Measurements spread farther from the hidden state. The filter lowers its gain, but uncertainty about the state generally increases.
3. **Same data, different assumptions.** Uncheck Match simulator. The actual q and r sliders now control the data. Keep them fixed and change assumed q or r. Underestimating q creates lag; underestimating r makes the filter follow noise and can make its uncertainty unjustifiably small.
4. **Abrupt change.** With the default small q, the +2 jump causes an innovation followed by gradual adaptation. A constant-q Kalman filter does not automatically interpret a large innovation as a regime change.
5. **Initial uncertainty.** Set m0 away from zero. Compare small and large p0. A confident initial estimate can resist the first measurements even when it is wrong.
6. **Same gain, different uncertainty.** In a matched model, multiply q, r, and p0 by the same factor. The gain sequence is unchanged, while the state and measurement scales and reported uncertainty change.
7. **Inspect one update.** Move the frame selector near the beginning, near the jump, or late in the sequence. Relate the numeric update to the gain and variance panels.

The identity observation map means the hidden state is the expected value of the chosen measurement. This experiment estimates that baseline; it does not refit AEther's local STBR decomposition. Its Gaussian, temporally independent observation-noise model would be an approximation for overlapping STBR histories and resonator outputs.

The simulated state starts at exactly zero, while the filter is initialized with the chosen uncertain prior. Confidence bands are conditional on the filter's assumed model. A mismatch, an abrupt jump, or an incorrect initial prior can make them misleading.
